# SageMath 10.9 — install test, and what Sage adds to the $Z$ work

Companion to `Partition_Function_01_Ising_Transfer_Matrix.ipynb`, which did everything **numerically**
with numpy. This notebook redoes the core of it **symbolically**.

The difference is the whole reason to have a CAS: numpy gave you `2.35040`; Sage gives you
$2\sinh\beta$.

In [1]:
print(version())
import sys; print('arch :', __import__('platform').machine())
print('BLAS ok:', __import__('numpy').linalg.eigvalsh(__import__('numpy').eye(512)).shape)

SageMath version 10.9, Release Date: 2026-05-04
arch : arm64
BLAS ok: (512,)


## 1. The transfer matrix, symbolically in $\beta$

In [2]:
beta, J, h = var('beta J h', domain='real')
T = matrix(SR, 2, 2, [exp(beta*(J+h)), exp(-beta*J), exp(-beta*J), exp(beta*(J-h))])
show(T)
print('trace :', T.trace().simplify_full())
print('det   :', T.det().simplify_full())

[e^((J + h)*beta)      e^(-J*beta)]
[     e^(-J*beta) e^((J - h)*beta)]

trace : (e^(J*beta + 2*beta*h) + e^(J*beta))*e^(-beta*h)
det   : (e^(4*J*beta) - 1)*e^(-2*J*beta)


## 2. Eigenvalues in closed form — compare with the notebook's numerics

In [3]:
T0 = T.subs(h==0, J==1)
ev = [e.simplify_full() for e in T0.eigenvalues()]
print('lambda_pm =', ev)
print()
for b in [0.2, 0.5, 1.0, 2.0, 3.0]:
    lp = max(N(e.subs(beta==b)) for e in ev); lm = min(N(e.subs(beta==b)) for e in ev)
    xi = 1/log(lp/lm)
    print(f'  beta={b:4.1f}   lam+={lp:9.5f}  lam-={lm:9.5f}   xi={N(xi):9.4f}')
print()
print('numpy notebook gave, e.g. beta=1.0:  lam+=3.08616  lam-=2.35040  xi=3.672')

lambda_pm = [(e^(2*beta) - 1)*e^(-beta), (e^(2*beta) + 1)*e^(-beta)]

  beta= 0.2   lam+=  2.04013  lam-=  0.40267   xi=   0.6163
  beta= 0.5   lam+=  2.25525  lam-=  1.04219   xi=   1.2954
  beta= 1.0   lam+=  3.08616  lam-=  2.35040   xi=   3.6719
  beta= 2.0   lam+=  7.52439  lam-=  7.25372   xi=  27.2960
  beta= 3.0   lam+= 20.13532  lam-= 20.03575   xi= 201.7140

numpy notebook gave, e.g. beta=1.0:  lam+=3.08616  lam-=2.35040  xi=3.672


## 3. What Sage can do that numpy cannot

The free energy per spin, **differentiated symbolically** to get energy and heat capacity — no
finite differences, no numerical noise.

In [4]:
f = -(1/beta)*log(2*cosh(beta))          # free energy per spin, h=0, J=1
u = diff(beta*f, beta).simplify_full()     # energy per spin
C = (beta^2 * diff(u, beta) * -1).simplify_full()
print('f    =', f)
print('u    =', u, '     <-- exactly -tanh(beta)')
print('C/k  =', C)
print()
print('check against the numpy notebook at beta=1.0:')
print(f'   u   Sage {N(u.subs(beta==1)):.5f}   numpy -0.76159')
print(f'   C/k Sage {N(C.subs(beta==1)):.5f}   numpy  0.41997')

f    = -log(2*cosh(beta))/beta
u    = -sinh(beta)/cosh(beta)      <-- exactly -tanh(beta)
C/k  = beta^2/cosh(beta)^2

check against the numpy notebook at beta=1.0:
   u   Sage -0.76159   numpy -0.76159
   C/k Sage 0.41997   numpy  0.41997


## 4. Exact $Z$ for finite $N$ — a polynomial, not a float

In [5]:
lam_p, lam_m = 2*cosh(beta), 2*sinh(beta)
for N_ in (4, 8, 16):
    Z = (lam_p^N_ + lam_m^N_).simplify_full()
    print(f'N={N_:2d}  Z(beta=1) = {N(Z.subs(beta==1)):.10e}')
print()
print('numpy notebook: N=4 1.2123293134e+02 | N=8 9.1604387226e+03 | N=16 6.8584536591e+07')
Z8 = (lam_p^8 + lam_m^8).expand().simplify_full()
print(); print('Z for N=8, expanded:'); show(Z8)

N= 4  Z(beta=1) = 1.2123293134e+2
N= 8  Z(beta=1) = 9.1604387226e+3
N=16  Z(beta=1) = 6.8584536591e+7

numpy notebook: N=4 1.2123293134e+02 | N=8 9.1604387226e+03 | N=16 6.8584536591e+07

Z for N=8, expanded:


256*cosh(beta)^8 + 256*sinh(beta)^8

## 5. Things that need a CAS — a quick tour

Exact integrals, arbitrary precision, symbolic linear algebra, number theory.

In [6]:
x, k = var('x k')
print('integral    :', integrate(exp(-x^2), x, -oo, oo))
print('zeta(2)     :', zeta(2), '=', N(zeta(2), digits=20))
print('pi 50 digits:', N(pi, digits=50))
print('factor      :', factor(2^127 - 1), '(Mersenne prime)')
print('is_prime    :', is_prime(2^127 - 1))
E = EllipticCurve([0,0,1,-1,0]); print('elliptic rank:', E.rank())
print('symbolic sum:', sum(1/k^2, k, 1, oo))

integral    : sqrt(pi)
zeta(2)     : 1/6*pi^2 = 1.6449340668482264365
pi 50 digits: 3.1415926535897932384626433832795028841971693993751
factor      : 170141183460469231731687303715884105727 (Mersenne prime)
is_prime    : True


elliptic rank: 1
symbolic sum: 1/6*pi^2


---
**If every cell above ran, SageMath 10.9 is correctly installed and integrated.**

The §2–§4 outputs should agree with the numpy notebook to every printed digit — same physics, but
Sage keeps it exact until you ask for a number.